# 05 · Colores de equipación de los 20 equipos

Saca los dos colores de la **camiseta** de la primera equipación de cada club, a partir de la ficha del
club en la Wikipedia en inglés (plantilla de equipación: `body1`, `leftarm1`, `rightarm1` y sus patrones
`pattern_b1`, `pattern_la1`, `pattern_ra1`). El pantalón y las medias no cuentan.

**Temporada.** Las fichas de hoy ya muestran la equipación 2026/27. Para la 2025/26 se usa la versión de cada
artículo a 31 de mayo de 2026, último día de la temporada, y se guarda su número de revisión.

**Reproducible sin conexión.** Lo descargado de Wikipedia (los campos de cada ficha con su revisión y su
fecha, las imágenes de los patrones y las siluetas de la plantilla) está guardado en
`data/wikitexto_equipaciones.json`, que se versiona. El cuaderno lee de ese fichero por defecto y da siempre
el mismo resultado. Solo vuelve a Wikipedia si se pone `DESCARGAR = True` (o la variable de entorno
`SCOUTING_DESCARGAR=1`).

**Cómo se miden los colores.** La plantilla de Wikipedia pinta cada parte (cuerpo, manga izquierda y manga
derecha) con su color de fondo y encima la imagen del patrón (rayas, franjas, ribetes). Aquí se hace lo
mismo, se recorta cada parte con la silueta de la camiseta y se cuentan los píxeles de cada color. Los colores
casi iguales (distancia RGB menor de 60, por el suavizado de bordes) se agrupan.

**Color primario.** El blanco no identifica a ningún equipo salvo al Real Madrid. Por eso el color primario
es el color de la camiseta con más superficie **que no sea blanco**, y el secundario es el otro de los dos
colores principales. Hay tres excepciones decididas por el autor, escritas abajo.

**Son colores aproximados.** Wikipedia usa colores genéricos (el mismo `#ff0000` para casi todos los rojos),
no los colores de marca oficiales de los clubes.

In [1]:
# Instalamos las librerías del cuaderno; si ya están instaladas no cambia nada
%pip install -q pandas requests pillow

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importamos os para situarnos en la raíz del proyecto
import os
# Importamos io para abrir imágenes desde memoria
import io
# Importamos re para leer la plantilla de la ficha
import re
# Importamos json para la caché
import json
# Importamos time para espaciar las consultas
import time
# Importamos base64 para guardar las imágenes dentro del JSON
import base64
# Importamos datetime para la fecha de consulta
import datetime
# Importamos pandas para la tabla de colores
import pandas as pd
# Importamos Pillow para las imágenes
from PIL import Image, ImageDraw, ImageFont

# Si el cuaderno se ejecuta desde notebooks/, subimos a la raíz para usar rutas relativas
if os.path.basename(os.getcwd()) == "notebooks":
    # Cambiamos a la carpeta superior
    os.chdir("..")

# Por defecto no se descarga nada: se lee la copia versionada
DESCARGAR = os.environ.get("SCOUTING_DESCARGAR") == "1"
# Copia versionada de lo descargado de Wikipedia
RUTA_CACHE = "data/wikitexto_equipaciones.json"
# Fecha de la versión de los artículos: último día de la temporada 2025/26
FECHA_VERSION = "2026-05-31T23:59:59Z"
# Fondo oscuro del panel, para la hoja de revisión
FONDO = "#0d1420"
# Campos de la plantilla de equipación que forman la camiseta de la primera equipación
CAMPOS = ["body1", "leftarm1", "rightarm1", "pattern_b1", "pattern_la1", "pattern_ra1"]
# Imagen de cada parte: (prefijo del fichero de patrón, silueta de la plantilla)
PARTES = {"b": ("Kit body", "File:Kit body.svg"), "la": ("Kit left arm", "File:Kit left arm.svg"),
          "ra": ("Kit right arm", "File:Kit right arm.svg")}
# Tamaño de cada parte en la plantilla de Wikipedia
TAMANO = {"b": (38, 59), "la": (31, 59), "ra": (31, 59)}

# Equipo del modelo y su entidad de Wikidata (los mismos del cuaderno 04)
CLUBES = {
    "Alaves": "Q223620", "Athletic Club": "Q8687", "Atletico Madrid": "Q8701", "Barcelona": "Q7156",
    "Celta Vigo": "Q8749", "Elche": "Q10512", "Espanyol": "Q8780", "Getafe": "Q8806", "Girona": "Q11945",
    "Levante": "Q8823", "Mallorca": "Q8835", "Osasuna": "Q10286", "Rayo Vallecano": "Q10300",
    "Real Betis": "Q8723", "Real Madrid": "Q8682", "Real Oviedo": "Q271574", "Real Sociedad": "Q10315",
    "Sevilla": "Q10329", "Valencia": "Q10333", "Villarreal": "Q12297",
}

## 1. Descarga de Wikipedia (solo si `DESCARGAR` es verdadero)

In [3]:
# Esta celda no hace nada en una ejecución normal
if DESCARGAR:
    # Importamos requests solo cuando hace falta
    import requests

    # Identificación del cliente, como piden las normas de uso de Wikimedia
    CABECERAS = {"User-Agent": "scouting-laliga-2526/1.0 (proyecto de datos; python-requests)"}
    # API de la Wikipedia en inglés
    API_EN = "https://en.wikipedia.org/w/api.php"

    # Petición GET con pausa y reintento si el servidor pide esperar
    def pedir(url, params=None, es_json=True):
        # Hasta 30 intentos
        for _ in range(30):
            # Petición
            r = requests.get(url, params=params, headers=CABECERAS, timeout=60)
            # Respuesta correcta
            if r.status_code == 200:
                # Pausa de cortesía
                time.sleep(1)
                # Devolvemos el contenido
                return r.json() if es_json else r.content
            # Solo se reintenta si el servidor pide esperar o falla
            if r.status_code != 429 and r.status_code < 500:
                r.raise_for_status()
            # Esperamos lo que diga el servidor
            time.sleep(int(r.headers.get("retry-after", 10)) + 1)
        # Sin respuesta
        raise RuntimeError(url)

    # Descarga una imagen de Wikipedia como PNG del ancho indicado, en base64; None si no existe
    def imagen_b64(titulo, ancho):
        # Información del fichero
        r = pedir(API_EN, {"action": "query", "titles": titulo, "prop": "imageinfo", "iiprop": "url",
                           "iiurlwidth": ancho, "format": "json", "formatversion": "2"})
        # Página del fichero
        pg = r["query"]["pages"][0]
        # Si no existe, no hay imagen
        if "imageinfo" not in pg:
            return None
        # Descargamos la versión rasterizada
        datos = pedir(pg["imageinfo"][0].get("thumburl") or pg["imageinfo"][0]["url"], es_json=False)
        # La guardamos como texto
        return base64.b64encode(datos).decode("ascii")

    # Título del artículo en la Wikipedia en inglés, desde Wikidata
    res = pedir("https://www.wikidata.org/w/api.php", {"action": "wbgetentities", "ids": "|".join(CLUBES.values()),
                                                      "props": "sitelinks", "sitefilter": "enwiki", "format": "json"})
    # Contenido de la caché
    nueva = {"fecha_version": FECHA_VERSION, "fecha_consulta": datetime.date.today().isoformat(),
             "siluetas": {}, "articulos": {}, "imagenes": {}}
    # Siluetas de la plantilla
    for parte, (_, silueta) in PARTES.items():
        # Contorno de cada parte
        nueva["siluetas"][parte] = imagen_b64(silueta, TAMANO[parte][0])
    # Recorremos los clubes
    for equipo, qid in CLUBES.items():
        # Título del artículo
        titulo = res["entities"][qid]["sitelinks"]["enwiki"]["title"]
        # Última versión del artículo anterior a la fecha
        p = pedir(API_EN, {"action": "query", "prop": "revisions", "titles": titulo, "rvlimit": 1,
                           "rvstart": FECHA_VERSION, "rvdir": "older", "rvprop": "ids|timestamp|content",
                           "rvslots": "main", "format": "json", "formatversion": "2"})
        # Revisión
        rev = p["query"]["pages"][0]["revisions"][0]
        # Texto del artículo
        texto = rev["slots"]["main"]["content"]
        # Líneas de la ficha con los campos de la camiseta, tal cual están en el wikitexto
        lineas = [m.group(0).strip() for c in CAMPOS for m in [re.search(r"\|\s*" + c + r"\s*=[^\n|}]*", texto)] if m]
        # Guardamos el artículo
        nueva["articulos"][equipo] = {"titulo": titulo, "url": "https://en.wikipedia.org/wiki/" + titulo.replace(" ", "_"),
                                      "revision": rev["revid"], "fecha_revision": rev["timestamp"][:10],
                                      "wikitexto_equipacion": lineas}
        # Imágenes de los patrones de las tres partes
        for parte, campo in [("b", "pattern_b1"), ("la", "pattern_la1"), ("ra", "pattern_ra1")]:
            # Valor del patrón en el wikitexto
            m = re.search(r"\|\s*" + campo + r"\s*=\s*([^\n|}]*)", texto)
            # Nombre del fichero del patrón
            fichero = f"File:{PARTES[parte][0]}{m.group(1).strip()}.png" if m and m.group(1).strip() else None
            # Lo descargamos una sola vez
            if fichero and fichero not in nueva["imagenes"]:
                nueva["imagenes"][fichero] = imagen_b64(fichero, TAMANO[parte][0])
        # Informamos
        print(equipo, rev["revid"], rev["timestamp"][:10])
    # Guardamos la caché
    json.dump(nueva, open(RUTA_CACHE, "w", encoding="utf-8"), ensure_ascii=False, indent=1)

## 2. Lectura de la copia versionada

In [4]:
# Copia de lo descargado de Wikipedia
cache = json.load(open(RUTA_CACHE, encoding="utf-8"))
# Fecha de consulta
print("consultado el", cache["fecha_consulta"], "· versión de los artículos a", cache["fecha_version"][:10])

# Ficha de cada club
fichas = []
for equipo, art in cache["articulos"].items():
    # Texto con las líneas de la equipación
    texto = "\n".join(art["wikitexto_equipacion"])
    # Valor de cada campo (vacío si no está)
    valores = {}
    for c in CAMPOS:
        # Busca "| campo = valor"
        m = re.search(r"\|\s*" + c + r"\s*=\s*([^\n|}]*)", texto)
        # Valor limpio
        valores[c] = m.group(1).strip() if m else ""
    # Guardamos la ficha
    fichas.append({"equipo": equipo, "articulo": art["url"], "revision": art["revision"],
                   "fecha_revision": art["fecha_revision"], "fecha_consulta": cache["fecha_consulta"], **valores})
# Pasamos a tabla
fichas = pd.DataFrame(fichas)
# Mostramos los campos leídos
print(fichas[["equipo", "revision", "fecha_revision"] + CAMPOS].to_string(index=False))


# Abre una imagen guardada en base64; None si no existe
def abrir(b64):
    # Sin imagen
    if not b64:
        return None
    # Imagen RGBA
    return Image.open(io.BytesIO(base64.b64decode(b64))).convert("RGBA")


# Máscara de cada parte: píxeles dentro de la silueta (transparentes en la imagen de contorno)
MASCARA = {}
for parte in PARTES:
    # Contorno de la plantilla, al tamaño de la parte
    contorno = abrir(cache["siluetas"][parte]).resize(TAMANO[parte], Image.NEAREST)
    # Dentro de la camiseta = alfa casi nulo en el contorno
    MASCARA[parte] = [a < 50 for a in contorno.getchannel("A").tobytes()]

consultado el 2026-11-16 · versión de los artículos a 2026-05-31
         equipo   revision fecha_revision  body1 leftarm1 rightarm1           pattern_b1          pattern_la1          pattern_ra1
         Alaves 1355951832     2026-05-24 0066e1   0066e1    0066e1         _alaves2526h         _alaves2526h         _alaves2526h
  Athletic Club 1356670834     2026-05-29 FFFFFF   FFFFFF    FFFFFF         _bilbao2627h         _bilbao2627h         _bilbao2627h
Atletico Madrid 1357137026     2026-05-31 FF0000   FF0000    FF0000 _atleticomadrid2526h _atleticomadrid2526h _atleticomadrid2526h
      Barcelona 1356877329     2026-05-30 14266e   14266e    14266e      _barcelona2526h      _barcelona2526h      _barcelona2526h
     Celta Vigo 1356706508     2026-05-29 A3D0F4   A3D0F4    A3D0F4          _celta2526h                                          
          Elche 1356630873     2026-05-28                                    _elche2526h          _elche2526h          _elche2526h
       Espanyol 13

## 3. Colores de la camiseta

In [5]:
# Convierte "FF0000" en (255, 0, 0); vacío = blanco, que es el fondo por defecto de la plantilla
def rgb(hexa):
    # Limpiamos el valor
    h = (hexa or "").strip().lstrip("#")
    # Sin valor, la plantilla usa blanco
    if not re.fullmatch(r"[0-9a-fA-F]{6}", h):
        return (255, 255, 255)
    # Tres canales
    return tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))


# Agrupa colores casi iguales y devuelve [(color, peso)] de mayor a menor
def agrupar(cuenta, umbral=60):
    # Grupos: [color representativo, píxeles]
    grupos = []
    # Recorremos los colores de más a menos frecuentes (a igualdad, por valor, para que sea determinista)
    for color, n in sorted(cuenta.items(), key=lambda x: (-x[1], x[0])):
        # Buscamos un grupo cercano
        for g in grupos:
            # Distancia euclídea en RGB
            if sum((a - b) ** 2 for a, b in zip(color, g[0])) ** 0.5 < umbral:
                # Sumamos al grupo
                g[1] += n
                break
        else:
            # Nuevo grupo
            grupos.append([color, n])
    # Ordenamos los grupos por superficie
    grupos.sort(key=lambda g: (-g[1], g[0]))
    # Total de píxeles
    total = sum(n for _, n in grupos)
    # Pesos en tanto por cien
    return [(c, n / total * 100) for c, n in grupos]


# Hexadecimal de un color
def hexa(c):
    # Formato #rrggbb
    return "#{:02x}{:02x}{:02x}".format(*c)


# Colores de la camiseta de cada equipo
filas = []
for f in fichas.itertuples():
    # Píxeles por color en toda la camiseta
    cuenta = {}
    # Recorremos cuerpo y mangas
    for parte, campo_color in [("b", "body1"), ("la", "leftarm1"), ("ra", "rightarm1")]:
        # Fondo de la parte con su color
        lienzo = Image.new("RGBA", TAMANO[parte], rgb(getattr(f, campo_color)) + (255,))
        # Patrón de la parte, si lo hay
        patron = getattr(f, "pattern_" + parte + "1")
        # Imagen del patrón
        im = abrir(cache["imagenes"].get(f"File:{PARTES[parte][0]}{patron}.png")) if patron else None
        # Lo pintamos encima del fondo
        if im is not None:
            lienzo.alpha_composite(im.resize(TAMANO[parte], Image.NEAREST))
        # Píxeles RGB de la parte
        datos = lienzo.convert("RGB").tobytes()
        # Contamos los de dentro de la silueta
        for i, dentro in enumerate(MASCARA[parte]):
            # Solo los de la camiseta
            if dentro:
                # Color del píxel
                px = tuple(datos[i * 3:i * 3 + 3])
                cuenta[px] = cuenta.get(px, 0) + 1
    # Colores agrupados
    grupos = agrupar(cuenta)
    # Fila de resultado: los dos colores con más superficie
    filas.append({"equipo": f.equipo, "wiki_color_1": hexa(grupos[0][0]), "peso_1": round(grupos[0][1], 1),
                  "wiki_color_2": hexa(grupos[1][0]), "peso_2": round(grupos[1][1], 1),
                  "otros": "; ".join(f"{hexa(c)} {p:.1f} %" for c, p in grupos[2:5])})
# Pasamos a tabla
colores = fichas.merge(pd.DataFrame(filas), on="equipo")
# Mostramos el resultado
print(colores[["equipo", "wiki_color_1", "peso_1", "wiki_color_2", "peso_2", "otros"]].to_string(index=False))

         equipo wiki_color_1  peso_1 wiki_color_2  peso_2                                         otros
         Alaves      #0066e1    50.7      #ffffff    37.0                 #0049a5 11.8 %; #000000 0.5 %
  Athletic Club      #ffffff    47.0      #ff0b0b    46.3   #191919 6.1 %; #700000 0.2 %; #717171 0.2 %
Atletico Madrid      #ffffff    52.3      #ff0000    46.4   #000000 0.6 %; #ff3e3e 0.4 %; #1f4dba 0.3 %
      Barcelona      #14266e    53.1      #ba002f    25.4  #891956 20.0 %; #000000 0.6 %; #ffffff 0.6 %
     Celta Vigo      #a3d0f4    89.1      #000000     4.0   #ffffff 3.4 %; #ff0000 3.2 %; #d497ac 0.4 %
          Elche      #ffffff    76.2      #008127    15.4   #45a261 7.4 %; #000000 0.6 %; #78bb8b 0.5 %
       Espanyol      #ffffff    58.0      #0000ff    41.0                  #000000 0.5 %; #dadada 0.5 %
         Getafe      #0000ff    88.2      #ff0000     5.7   #ffff00 4.0 %; #ffffff 1.6 %; #000000 0.5 %
         Girona      #ffffff    83.9      #ff0000    15.1       

## 4. Color primario y secundario

- **Regla:** de los dos colores con más superficie, el primario es el que no es blanco; si ninguno es blanco,
  el de más superficie. El secundario es el otro.
- **Excepciones del autor:**
  - **Real Madrid:** blanco. Es el único equipo al que el blanco identifica.
  - **Barcelona:** el grana (`#ba002f`), que es su segundo color de camiseta, en lugar del azul.
  - **Valencia:** su camiseta es blanca y el blanco ya es del Real Madrid. Se usa el naranja dominante de su
    escudo (`#e33e07`, cuaderno 03). Es el único color que no sale de la plantilla de Wikipedia.
- `color_origen` vale `wikipedia_equipacion` si el color primario es uno de los colores de camiseta medidos
  en Wikipedia, y `revision_autor` si no lo es.

In [6]:
# Excepciones decididas por el autor (David Palacios, 16/11/2026)
EXCEPCIONES_AUTOR = {"Real Madrid": "#ffffff", "Barcelona": "#ba002f", "Valencia": "#e33e07"}


# Decide primario, secundario y origen de un equipo
def decidir(f):
    # Los dos colores de camiseta de Wikipedia
    c1, c2 = f["wiki_color_1"], f["wiki_color_2"]
    # Regla: el que no es blanco; si ninguno lo es, el de más superficie
    primario = c2 if c1 == "#ffffff" else c1
    # Excepción del autor, si la hay
    primario = EXCEPCIONES_AUTOR.get(f["equipo"], primario)
    # El secundario es el otro color de la camiseta; si el primario no es de la camiseta, el principal
    secundario = c2 if primario == c1 else c1
    # Origen: de la plantilla de Wikipedia o revisión del autor
    origen = "wikipedia_equipacion" if primario in (c1, c2) else "revision_autor"
    # Devolvemos los tres valores
    return primario, secundario, origen


# Aplicamos a los 20
colores[["color_primario", "color_secundario", "color_origen"]] = [decidir(f) for _, f in colores.iterrows()]

# Lista de colores primarios confirmada por el autor: el resultado tiene que coincidir exactamente
CONFIRMADO = {
    "Alaves": "#0066e1", "Athletic Club": "#ff0b0b", "Atletico Madrid": "#ff0000", "Barcelona": "#ba002f",
    "Celta Vigo": "#a3d0f4", "Elche": "#008127", "Espanyol": "#0000ff", "Getafe": "#0000ff", "Girona": "#ff0000",
    "Levante": "#661424", "Mallorca": "#ff0000", "Osasuna": "#ff0000", "Rayo Vallecano": "#ff0000",
    "Real Betis": "#349b5c", "Real Madrid": "#ffffff", "Real Oviedo": "#0000dd", "Real Sociedad": "#475cad",
    "Sevilla": "#e20000", "Valencia": "#e33e07", "Villarreal": "#ffff00",
}
# Si algo no coincide, paramos
assert dict(zip(colores["equipo"], colores["color_primario"])) == CONFIRMADO, "los colores no coinciden con la lista confirmada"

# Color dominante del escudo, en columna aparte para comparar
colores = colores.merge(pd.read_csv("data/escudos.csv")[["equipo", "fichero", "color_primario"]].rename(
    columns={"color_primario": "color_escudo"}), on="equipo")
# Mostramos el resultado
print(colores[["equipo", "color_primario", "color_secundario", "color_origen", "color_escudo"]].to_string(index=False))

# Colores repetidos: por qué el color no identifica al equipo
print("\nequipos por color primario:")
print(colores.groupby("color_primario")["equipo"].agg(lambda s: ", ".join(s)).to_string())

         equipo color_primario color_secundario         color_origen color_escudo
         Alaves        #0066e1          #ffffff wikipedia_equipacion      #0033a0
  Athletic Club        #ff0b0b          #ffffff wikipedia_equipacion      #e11f1c
Atletico Madrid        #ff0000          #ffffff wikipedia_equipacion      #282a6f
      Barcelona        #ba002f          #14266e wikipedia_equipacion      #deb500
     Celta Vigo        #a3d0f4          #000000 wikipedia_equipacion      #ff0000
          Elche        #008127          #ffffff wikipedia_equipacion      #e0cf9c
       Espanyol        #0000ff          #ffffff wikipedia_equipacion      #e01517
         Getafe        #0000ff          #ff0000 wikipedia_equipacion      #0062a5
         Girona        #ff0000          #ffffff wikipedia_equipacion      #d02030
        Levante        #661424          #222730 wikipedia_equipacion      #fcc60c
       Mallorca        #ff0000          #000000 wikipedia_equipacion      #ffe600
        Osasuna 

color_primario
#0000dd                                          Real Oviedo
#0000ff                                     Espanyol, Getafe
#0066e1                                               Alaves
#008127                                                Elche
#349b5c                                           Real Betis
#475cad                                        Real Sociedad
#661424                                              Levante
#a3d0f4                                           Celta Vigo
#ba002f                                            Barcelona
#e20000                                              Sevilla
#e33e07                                             Valencia
#ff0000    Atletico Madrid, Girona, Mallorca, Osasuna, Ra...
#ff0b0b                                        Athletic Club
#ffff00                                           Villarreal
#ffffff                                          Real Madrid


**El color no identifica al equipo en este panel.** Con este criterio hay 7 equipos rojos y 5 azules, y
cuatro comparten exactamente el mismo `#ff0000`. Al equipo lo identifica el escudo. El color sirve para
acentuar la ficha del jugador seleccionado, donde solo hay un equipo en pantalla.

## 5. Guardado e imagen de revisión

In [7]:
# Guardamos la tabla de colores, que lee el cuaderno 01
colores.drop(columns=["fichero"]).to_csv("data/colores_equipos.csv", index=False)

# Fuente legible, si el sistema la tiene
try:
    # Fuente de Windows
    letra, letra_peq = ImageFont.truetype("segoeui.ttf", 20), ImageFont.truetype("segoeui.ttf", 15)
except OSError:
    # Fuente por defecto de Pillow
    letra, letra_peq = ImageFont.load_default(20), ImageFont.load_default(15)


# Luminancia relativa (WCAG), solo para elegir tinta blanca o negra sobre cada rectángulo
def luminancia(c):
    # Canal linealizado
    lin = [(v / 255 / 12.92) if v / 255 <= 0.03928 else ((v / 255 + 0.055) / 1.055) ** 2.4 for v in c]
    # Combinación de canales
    return 0.2126 * lin[0] + 0.7152 * lin[1] + 0.0722 * lin[2]


# La hoja necesita los 20 escudos, que no se versionan todos; si faltan, no se dibuja
if all(os.path.exists(os.path.join("images/escudos", f)) for f in colores["fichero"]):
    # Alto de cada fila y ancho de la imagen
    ALTO, ANCHO = 84, 1000
    # Lienzo con el fondo del panel
    hoja = Image.new("RGBA", (ANCHO, ALTO * 20 + 50), FONDO)
    # Lápiz
    d = ImageDraw.Draw(hoja)
    # Cabecera
    for x, texto in [(20, "Escudo · equipo"), (330, "Primario"), (560, "Secundario"), (790, "Origen del primario")]:
        # Título de la columna
        d.text((x, 14), texto, font=letra_peq, fill="#9aa4b2")
    # Recorremos los equipos
    for n, f in enumerate(colores.sort_values("equipo").itertuples()):
        # Arriba de la fila
        y = 50 + n * ALTO
        # Escudo a 64 px
        hoja.alpha_composite(Image.open(os.path.join("images/escudos", f.fichero)).resize((64, 64), Image.LANCZOS), (20, y + 8))
        # Nombre del equipo
        d.text((100, y + 28), f.equipo, font=letra, fill="#e6e9ee")
        # Rectángulos de los dos colores con su hexadecimal
        for x, color in [(330, f.color_primario), (560, f.color_secundario)]:
            # Rectángulo del color
            d.rectangle([x, y + 10, x + 200, y + 72], fill=color, outline="#5a6475", width=1)
            # Texto en blanco o negro según el fondo
            d.text((x + 10, y + 28), color, font=letra, fill="#000000" if luminancia(rgb(color)) > 0.35 else "#ffffff")
        # Origen del color primario, resaltado si es revisión del autor
        d.text((790, y + 30), f.color_origen, font=letra_peq, fill="#9aa4b2" if f.color_origen == "wikipedia_equipacion" else "#ffb454")
        # Línea separadora
        d.line([20, y + ALTO - 2, ANCHO - 20, y + ALTO - 2], fill="#1f2a3a")
    # Guardamos
    hoja.save("images/revision_colores.png")
    # Informamos
    print("hoja de revisión guardada en images/revision_colores.png")
else:
    # Sin escudos no hay hoja
    print("faltan escudos en images/escudos/ (ejecuta antes el cuaderno 03): no se dibuja la hoja de revisión")

hoja de revisión guardada en images/revision_colores.png
